### Chains Parallel - Practice & Exercises

In this notebook, you will practice executing multiple chains concurrently using `RunnableParallel` and managing variables using `RunnablePassthrough`.

#### Part 1: Follow-Along Practice

##### Method 1: RunnableParallel

##### Goal / Question
List the main features of a product, then trigger parallel prompt chains to evaluate the pros and cons of those features separately. Finally, merge both review branches into a unified report.

##### Sample Input
```python
{"product_name": "MacBook Pro"}
```

##### Sample Output
A combined text report containing a "Pros:" evaluation and a "Cons:" evaluation.

##### Plan / Steps
1. Retrieve features of the product using a primary prompt-model chain.
2. Prepare a Pros Analysis chain: takes features as input and outputs pros list.
3. Prepare a Cons Analysis chain: takes features as input and outputs cons list.
4. Execute both analysis chains concurrently on the retrieved features dictionary using `RunnableParallel`.
5. Pipe the parallel outputs into a merge lambda to combine them into a single report.

In [ ]:
from dotenv import load_dotenv
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableParallel, RunnableLambda
from langchain_openai import ChatOpenAI

load_dotenv()
model = ChatOpenAI(model="gpt-4o")

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert product reviewer."),
    ("human", "List the main features of the product {product_name}."),
])

def analyze_pros(features):
    pros_template = ChatPromptTemplate.from_messages([
        ("system", "You are an expert product reviewer."),
        ("human", "Given these features: {features}, list the pros of these features."),
    ])
    return pros_template.format_prompt(features=features)

def analyze_cons(features):
    cons_template = ChatPromptTemplate.from_messages([
        ("system", "You are an expert product reviewer."),
        ("human", "Given these features: {features}, list the cons of these features."),
    ])
    return cons_template.format_prompt(features=features)

def combine_pros_cons(pros, cons):
    return f"Pros:\n{pros}\n\nCons:\n{cons}"

pros_branch_chain = RunnableLambda(lambda x: analyze_pros(x)) | model | StrOutputParser()
cons_branch_chain = RunnableLambda(lambda x: analyze_cons(x)) | model | StrOutputParser()

chain = (
    prompt_template
    | model
    | StrOutputParser()
    | RunnableParallel(branches={"pros": pros_branch_chain, "cons": cons_branch_chain})
    | RunnableLambda(lambda x: combine_pros_cons(x["branches"]["pros"], x["branches"]["cons"]))
)
result = chain.invoke({"product_name": "MacBook Pro"})
print(result)

##### Method 2: RunnablePassthrough

##### Goal / Question
Pass raw string input parameters through unchanged while dynamically binding other static values to build prompt template inputs.

##### Sample Input
```python
"artificial intelligence"
```

##### Sample Output
A short poem about artificial intelligence written in Shakespearean English style.

##### Plan / Steps
1. Define a template prompt requiring both `{topic}` and `{style}` variables.
2. Set up a mapping dictionary where `topic` uses `RunnablePassthrough()` and `style` maps to a lambda returning the style string.
3. Chain mapping, prompt, model, and output parser.
4. Invoke passing the raw string topic.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("Write a short poem about {topic} in the style of {style}.")

mapping_chain = {
    "topic": RunnablePassthrough(),
    "style": lambda x: "Shakespearean English"
}

chain_pt = mapping_chain | prompt | model | StrOutputParser()
print(chain_pt.invoke("artificial intelligence"))

#### Part 2: Practice Exercises

##### Exercise: City Guide Generator

##### Goal / Question
Given a city name, create a parallel chain where Branch 1 lists the top 3 restaurants, and Branch 2 lists the top 3 tourist attractions. Merge them into a single final description.

##### Sample Input
```python
{"city": "Paris"}
```

##### Sample Output
Food Recommendations:
[List of restaurants]

Attraction Recommendations:
[List of tourist places]

##### Plan / Steps
1. Define a template `restaurants_prompt` asking for 3 restaurants in `{city}`.
2. Define a template `attractions_prompt` asking for 3 attractions in `{city}`.
3. Create individual sub-chains: `restaurants_chain` and `attractions_chain` using model and string parser.
4. Combine them using `RunnableParallel(restaurants=restaurants_chain, attractions=attractions_chain)`.
5. Pipe the parallel outputs into a merge `RunnableLambda` that formats them into a final description.
6. Invoke the pipeline with a city name.

In [ ]:
# TODO: Write your implementation here

<details>
<summary>💡 Click to see solution</summary>

```python
restaurants_prompt = ChatPromptTemplate.from_template("List the top 3 places to eat in {city}.")
attractions_prompt = ChatPromptTemplate.from_template("List the top 3 tourist attractions in {city}.")

restaurants_chain = restaurants_prompt | model | StrOutputParser()
attractions_chain = attractions_prompt | model | StrOutputParser()

parallel_guide = RunnableParallel(restaurants=restaurants_chain, attractions=attractions_chain)
merge_guide = RunnableLambda(lambda x: f"Food Recommendations:\n{x[restaurants]}\n\nAttraction Recommendations:\n{x[attractions]}")

city_chain = parallel_guide | merge_guide
print(city_chain.invoke({"city": "Paris"}))
```
</details>